<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_post_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review post processing

## Set up

In [24]:
# @title Dependencies

# Installations
!pip install -q pandas tqdm

# First-party imports
from google.colab import drive
import os
import json

# Third-party imports
import pandas as pd

In [25]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the judged and parsed llm reviews
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_parsed")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_parsed.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_parsed.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_parsed.


In [26]:
# @title Setup definitions

TARGET_COLUMN = "llm_raw_response"

# Error messages from review generation/parsing
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING = "ERROR: UNKNOWN STATE REACHED IN PARALLEL PROCESSING"
EXTRACTION_FAILED_EMPTY_SECTION = "ERROR: EXTRACTION FAILED / SECTION MISSING"

# Error messages from review judgement generation/parsing
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."
EXTRACTION_ERROR = "Error: Failed to extract JSON from raw judgement"
PARSING_ERROR = "Error: Failed to parse judgement from JSON to single values"

# Define expected judgement labels
EXPECTED_LABEL_KEYS = [
    'actionability_label',
    'grounding_specificity_label',
    'verifiability_label',
    'helpfulness_label'
]

In [27]:
# @title Data import

# Read processed llm reviews
df = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_evaluated.parquet"))

# Check df
display(df.shape)
display(df.head(3))


(2, 10)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segment_hash,config_group,llm_raw_response
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,Experimental details in the notes are summariz...,ab3ce65ba319,model: gpt-5-mini-2025-08-07 | temperature: na...,"_label"": ""[GROUNDING SPECIFICITY LABEL]"",\n ""..."
1,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,The optimal 2:4 MVUE is reported to be computa...,4889f4e5fd66,model: gpt-5-mini-2025-08-07 | temperature: na...,"###Actionability Label: ""1""\n###Grounding Spe..."


## Define

Original regex-based output checks from the authors `inference_utils.py` to enforce a clean final JSON-output structure: https://github.com/bodasadallah/RevUtil/blob/main/inference/inference_utils.py (note that some function of their code do not apply to the present usecase, i.e. `known_keys`, `expected_keys`, `expected_valid_json()` and `get_gold_labels()`).

In [28]:
# @title inference_utils.py

import json
import re


known_keys = ['actionability_rationale', 'actionability_label', 'grounding_specificity_rationale', 'grounding_specificity_label', 'verifiability_rationale', 'verifiability_label', 'helpfulness_rationale', 'helpfulness_label']


def replace_category_names(text):
    category_mapping = {
        "1: Unverifiable": 1,
        "2: Borderline Verifiable": 2,
        "3: Somewhat Verifiable": 3,
        "4: Mostly Verifiable": 4,
        "5: Fully Verifiable": 5,
        "X: No Claim": "X",

        "1: Unactionable": 1,
        "2: Borderline Actionable": 2,
        "3: Somewhat Actionable": 3,
        "4: Mostly Actionable": 4,
        "5: Highly Actionable": 5,

        "1: Not Grounded": 1,
        "2: Weakly Grounded and Not Specific": 2,
        "3: Weakly Grounded and Specific": 3,
        "4: Fully Grounded and Under-Specific": 4,
        "5: Fully Grounded and Specific": 5,

        "1: Not Helpful at All": 1,
        "2: Barely Helpful": 2,
        "3: Somewhat Helpful": 3,
        "4: Mostly Helpful": 4,
        "5: Highly Helpful": 5
    }

    # Normalize dictionary for case-insensitive matching
    category_mapping_lower = {k.lower(): v for k, v in category_mapping.items()}
    partial_mapping_lower = {k.split(": ", 1)[-1].lower(): v for k, v in category_mapping.items()}

    def replace_match(match):
        matched_text = match.group(0).lower()  # Normalize case
        return str(category_mapping_lower.get(matched_text, partial_mapping_lower.get(matched_text, match.group(0))))

    # Replace full matches first (case-insensitive)
    pattern = re.compile(r'\b(?:' + '|'.join(map(re.escape, category_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern.sub(replace_match, text)

    # Replace partial matches (case-insensitive)
    pattern_partial = re.compile(r'\b(?:' + '|'.join(map(re.escape, partial_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern_partial.sub(replace_match, text)

    return text


expected_keys = [
    "actionability_rationale",
    "actionability_label",
    "grounding_specificity_rationale",
    "grounding_specificity_label",
    "verifiability_rationale",
    "verifiability_label",
    "helpfulness_rationale",
    "helpfulness_label"
]

import json
import re

import json

def extract_valid_json(text):
    label_keys = [
        "actionability_label",
        "grounding_specificity_label",
        "verifiability_label",
        "helpfulness_label"
    ]

    # Initialize result with None values
    result = {key: None for key in label_keys}

    # Match patterns like: "key": "1", "key": 1, or "key": "X"
    pattern = r'"(' + '|'.join(label_keys) + r')"\s*:\s*"?(X|\d+)"?'

    matches = re.findall(pattern, text)

    for key, val in matches:
        result[key] = str(val)  # Always store as string for valid JSON output

    return json.dumps(result, indent=2)



def escape_inner_quotes(text):
    """Finds specified rationale fields and escapes only internal double quotes."""
    fields = [
        "actionability_rationale",
        "grounding_specificity_rationale",
        "verifiability_rationale",
        "helpfulness_rationale"
    ]

    for field in fields:
        pattern = fr'("{field}"\s*:\s*")(.*?)("[\}},])'  # Escape closing brace
        matches = list(re.finditer(pattern, text, re.DOTALL))  # Find all matches first

        for match in reversed(matches):  # Process from last to first to avoid index shifting
            before, rationale, after = match.groups()
            escaped_rationale = rationale.replace('"', '\\"')  # Escape only inner quotes
            text = text[:match.start(2)] + escaped_rationale + text[match.end(2):]

    return text
def extract_dict(text):

    text = replace_category_names(text)  # Replace category names with numbers
    ## remove double spaces
    text = re.sub(' +', ' ', text)
    ## remove ``` from the text
    # text = text.replace('```', '')
    text = text.replace("-", "")  # Remove leading hyphens
    text = text.replace("\n", " ")  # Remove newlines
    text = text.replace("\\'", "'")  # Fix incorrectly escaped single quotes
    text = text.replace('\\"s', "'s")  # Fix incorrect escaped possessive 's
    text = text.replace("\\\\'", "\\\"")
    text = text.replace("\\\\", "\\")

    text = text.replace("[", "")  # Remove square brackets
    text = text.replace("]", "")  # Remove square brackets

    ############## For Prometheus2 #################
    # text = text.replace("[", '"')  # Replace single quotes with double quotes
    # text = text.replace("]", '"')  # Replace single quotes with double quotes

    ## if text begin with comma or space, remove it
    if text[0] == ',' or text[0] == ' ':
        text = text[1:]

    text = text.replace("\\\\", "\\") # Fix double backslashes
    dict_str  = ""
    if "```" in text:
        text = text + '#'
        match = re.search(r"```(?:json)?\s*(.*?)(```)?#", text, re.DOTALL)
        if match:
            text = match.group(0)
            ## remove the ```json  and ``` from the text
            text = text.replace('```json', '')
            text = text.replace('```', '')
            text = text.replace('#', '')

    text = text.strip()  # Remove leading and trailing whitespace

    if not text:
        return None

    ############# Comment for Prometheus2 ##############
    if text[0] != '{':
        text = '{' + text + '}'

    ################## cut the text if there is two newlines. This is for Prmetheus2 #########
    # if '\n\n' in text:
    #     halfs = text = text.split('\n\n', 1)
    #     text = halfs[0] if "actionability_label" in halfs[0] else halfs[1]

    if '{' not in text:
        text = '{' + text + '}'

    text = text.replace(" }  { ", ',')  # Remove newlines between dictionaries

    text = text.replace("\n", ' ')  # Remove newlines

    ############################# Prometheus2 ##########################
    # text = extract_valid_json(text)
    # text = text.replace('\\', '')  # Remove single quotes
    # print(f"Text after processing: {text} \n\n\n")

    ############ Some cases doesn't work with replacing the quotes, so trying both ways
    text2 = text
    try:
        text = text.replace("'", '"')  # Replace single quotes with double quotes
        text = escape_inner_quotes(text)  # Fix quotes inside rationale fields
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        return json.loads(dict_str)  # Convert to Python dictionary safely
    except json.JSONDecodeError:
        print("Replacing quotes didn't work, trying without replacing quotes.")
        text = text2  # Revert to original text
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        try:
            return json.loads(dict_str)  # Convert to Python dictionary safely
        except json.JSONDecodeError as e:
            print(f"Parsing error: {e}\nProblematic string: {dict_str}")
            return None

def extract_predictions(model_outputs):
    """
    Parses a list of model-generated texts to extract labels and returns a dictionary.

    :param model_outputs: List of strings containing model-generated text with labels.
    :return: List of dictionaries with extracted labels.
    """
    extracted_data = []

    for text in model_outputs:

        if 'outputs' in text.keys():
            text = text.outputs[0].text
        elif 'generated_text' in text.keys():
            text = text['generated_text']

        extracted_dict = extract_dict(text)
        if  not extracted_dict:
            extracted_data.append({
                'actionability_label': None,
                'grounding_specificity_label': None,
                'verifiability_label': None,
                'helpfulness_label': None,
                'actionability_rationale': None,
                'grounding_specificity_rationale': None,
                'verifiability_rationale': None,
                'helpfulness_rationale': None
            })
            continue

        parsed_result = {
            'actionability_label': str(extracted_dict.get('actionability_label', None)),
            'grounding_specificity_label':  str(extracted_dict.get('grounding_specificity_label', None)),
            'verifiability_label':  str(extracted_dict.get('verifiability_label', None)),
            'helpfulness_label':  str(extracted_dict.get('helpfulness_label', None)),
            ### rationale keys
            'actionability_rationale':  str(extracted_dict.get('actionability_rationale', None)),
            'grounding_specificity_rationale':  str(extracted_dict.get('grounding_specificity_rationale', None)),
            'verifiability_rationale':  str(extracted_dict.get('verifiability_rationale', None)),
            'helpfulness_rationale':  str(extracted_dict.get('helpfulness_rationale', None))
        }

        extracted_data.append(parsed_result)

    return extracted_data







aspects = [ 'actionability', 'grounding_specificity','verifiability', 'helpfulness']
def get_gold_labels(raw_data, dataset_config,aspect_row_name='chatgpt_ASPECT_score'):

    gold_labels = []
    dataset_config = aspects if dataset_config == 'all' else dataset_config
    if type(dataset_config) == str:
        dataset_config = [dataset_config]

    for row in raw_data:
        row_data = {}
        for aspect in dataset_config:
            row_name = aspect_row_name.replace('ASPECT', aspect)
            if row_name in row:
                row_data[aspect] = row[row_name]
            else:
                row_data[aspect] = None
        gold_labels.append(row_data)

    return gold_labels

In [29]:
def unified_parsing_function(raw_llm_output: str) -> dict:
    """
    A wrapper function that uses extract_predictions() to parse the values out of the target string
    """
    result_dict = {key: None for key in EXPECTED_LABEL_KEYS}

    # List of known error strings that might appear in the raw output from previous stages
    known_error_strings = [
        JUDGE_GENERATION_FAILURE,
        EXTRACTION_ERROR,
        PARSING_ERROR,
        UNKNOWN_ERROR_STATE
    ]

    # 1. Check if the raw output is an existing error string. If so, propagate it directly.
    if raw_llm_output in known_error_strings:
        for key in EXPECTED_LABEL_KEYS:
            result_dict[key] = raw_llm_output
        return result_dict

    # 2. Prepare input for extract_predictions (expects a list of objects)
    mock_model_output_list = [{'generated_text': raw_llm_output}]

    # 3. Extract predictions
    extracted_preds = extract_predictions(mock_model_output_list)

    # 4. Check extracted content

    # If extracted list is completly empty
    if not extracted_preds:
        # Assign all keys the error string
        for key in EXPECTED_LABEL_KEYS:
            result_dict[key] = EXTRACTION_ERROR
        return result_dict

    # Extract the (only) dictionary from the list
    parsed_data = extracted_preds[0]

    # Check for completly empty keys
    all_labels_none = True
    for key in EXPECTED_LABEL_KEYS:
        if parsed_data.get(key) is not None:
            all_labels_none = False
            break
    if all_labels_none:
        for key in EXPECTED_LABEL_KEYS:
            # Assign all keys the error string
            result_dict[key] = EXTRACTION_ERROR
        return result_dict

    # 5. Populate result_dict and map None values to error string
    for key in EXPECTED_LABEL_KEYS:
        value = parsed_data.get(key)
        if value is None:
            # Assign the value of the key the parsing error string
            result_dict[key] = PARSING_ERROR
        else:
            # Convert the value of the key
            result_dict[key] = str(value)

    return result_dict

## Run

In [30]:
# @title Execution

# Apply the parsing function to the target column
parsed_results = df[TARGET_COLUMN].apply(unified_parsing_function)

# Convert the list of dictionaries into a DataFrame of new columns
parsed_df = pd.DataFrame(parsed_results.tolist())

# Concatenate the new columns with the original DataFrame
df_parsed = pd.concat([df, parsed_df], axis=1)

# Check and display
display(df_parsed.shape)
display(df_parsed.head(3))

Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {_label": "GROUNDING SPECIFICITY LABEL",  "verifiability_label": "VERIFIABILITY LABEL",  "helpfulness_label": "HELPFULNESS LABEL" }
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {###Actionability Label: "1" ###Grounding Specificity Label: "1" ###Verifiability Label: "1" ###Helpfulness Label: "1" ###Review Point: The proposed approxMVUE is a heuristic with a variance bound but the practical impact on wallclock training time depends heavily on hardware/software support and on implementation optimization — the paper cannot demonstrate actual speedups because current accelerators lack training support for sparse GEMMs, so measured speedups are inferred rather than observed. ###Output: {  "acti

(2, 14)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segment_hash,config_group,llm_raw_response,actionability_label,grounding_specificity_label,verifiability_label,helpfulness_label
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,Experimental details in the notes are summariz...,ab3ce65ba319,model: gpt-5-mini-2025-08-07 | temperature: na...,"_label"": ""[GROUNDING SPECIFICITY LABEL]"",\n ""...",Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement
1,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,The optimal 2:4 MVUE is reported to be computa...,4889f4e5fd66,model: gpt-5-mini-2025-08-07 | temperature: na...,"###Actionability Label: ""1""\n###Grounding Spe...",Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement,Error: Failed to extract JSON from raw judgement


## Transformation

In [31]:
# @title Results

# Save the long-format DataFrame to JSONL
df_parsed.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"DataFrame saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format DataFrame to Parquet
df_parsed.to_parquet(RESULTS_PATH, index=False)
print(f"DataFrame saved to Parquet at: {RESULTS_PATH}")

DataFrame saved to JSONL at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_parsed/llm_reviews_parsed.jsonl
DataFrame saved to Parquet at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_parsed/llm_reviews_parsed.parquet
